# Final Project

**AM 115 Mathematical Modeling**

_Eric Ordonez, 5/8/2026_

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import networks
import solver
import utils
from pathlib import Path
from params import KAPPA, THETA

In [ ]:
IMAGES = Path('./images')
IMAGES.mkdir(exist_ok=True)

RESULTS = Path('./results')
RESULTS.mkdir(exist_ok=True)

plt.style.use('seaborn-v0_8-white')
pd.options.display.float_format = '{:.4f}'.format

## Stylized networks

In [ ]:
# Grid network.
G_grid, cordon_grid = networks.build_grid_network()
grid_pos = {
    node: ((node - 1) % 3, -((node - 1) // 3))
    for node in G_grid.nodes()
}
grid_node_colors = utils.node_colors(G_grid, cordon_grid)

# Hub-and-spoke network.
G_hub, cordon_hub = networks.build_hub_network()
hub_pos = nx.spring_layout(G_hub, seed=2050)
hub_node_colors = utils.node_colors(G_hub, cordon_hub)

# Core-periphery network.
G_core, cordon_core = networks.build_core_network()
core_pos = nx.spring_layout(G_core, seed=205)
core_node_colors = utils.node_colors(G_core, cordon_core)

In [ ]:
# Plot the networks in one figure.
fig_topologies, axes = plt.subplots(ncols=3, figsize=(12, 4.8))

utils.plot_network(G_grid, grid_pos, grid_node_colors,
                   ax=axes[0],
                   transit_offset=0.02,
                   node_size=400,
                   title='')
utils.plot_network(G_hub, hub_pos, hub_node_colors,
                   ax=axes[1],
                   transit_offset=0.015,
                   node_size=400,
                   title='')
utils.plot_network(G_core, core_pos, core_node_colors,
                   ax=axes[2],
                   transit_offset=0.012,
                   node_size=400,
                   title='(c) Core-periphery')

axes[0].set_title('(a) Grid', size='x-large', pad=15)
axes[1].set_title('(b) Hub-and-spoke', size='x-large', pad=15)
axes[2].set_title('(c) Core-periphery', size='x-large', pad=15)

fig_topologies.tight_layout()
fig_topologies.savefig(IMAGES / 'network_topologies.png')

## Simulations

### Calibrate demand

In [ ]:
def find_best_q(q_sweep, G, W, paths, kappa=KAPPA):
    max_ratios = []
    for q in q_sweep:
        print(f'q={q}\t', end='')
        Q = {od: q for od in W}
        x, f = solver.solve_sue(G, paths, W, Q, theta=1.0, kappa=kappa)
        max_ratio = 0
        road_edges = [
            ((u, v), d)
            for u, v, k, d in G.edges(keys=True, data=True) if k == 'car'
        ]
        for edge, data in road_edges:
            ratio = x['car'][edge] / data['K']
            max_ratio = max(ratio, max_ratio)
        max_ratios.append(max_ratio)

    print("====== Finding best q ======")
    max_ratios = np.asarray(max_ratios)
    if np.all(max_ratios > 1):
        print(f'All ratios greater than 1, lowest={max_ratios[0]}')
    elif np.all(max_ratios < 1):
        print(f'All ratios less than 1, highest={max_ratios[-1]}')
    else:
        idx = np.argmin(max_ratios < 1) - 1
        print(f'q={q_sweep[idx]}\tmax x/K={max_ratios[idx]}')

In [ ]:
# OD pairs and paths.
W_grid = [
    (1,9), (9,1), (3,7), (7,3),
    (1,6), (6,1), (3,4), (4,3),
    (2,8), (8,2), (1,8), (8,1)
]
grid_paths = networks.build_paths(G_grid, cordon_grid, W_grid)

W_hub = [
    (1,9), (9,1), (2,8), (8,2), # Hub-crossing
    (4,7), (7,4), (4,6), (6,4), # Hub-crossing
    (1,3), (3,1), (7,9), (9,7), # Periphery
]
hub_paths = networks.build_paths(G_hub, cordon_grid, W_hub)

W_core = [
    (1,9), (9,1), (2,8), (8,2),
    (3,7), (7,3), (1,5), (5,1),
    (3,9), (9,3)
]
core_paths = networks.build_paths(G_core, cordon_grid, W_core)

In [ ]:
print("GRID NETWORK")
print("============")
q_sweep = np.linspace(6.5, 6.6, 6)
find_best_q(q_sweep, G_grid, W_grid, grid_paths)

In [ ]:
print("HUB-AND-SPOKE NETWORK")
print("=====================")
q_sweep = np.linspace(9.3, 9.4, 6)
find_best_q(q_sweep, G_hub, W_hub, hub_paths)

In [ ]:
print("CORE-PERIPHERY NETWORK")
print("======================")
q_sweep = np.linspace(5.4, 5.6, 5)
find_best_q(q_sweep, G_core, W_core, core_paths)

In [ ]:
# Demand such that there is already saturating congestion.
Q_grid = {od: 6.52 for od in W_grid}
Q_hub =  {od: 9.34 for od in W_hub}
Q_core = {od: 5.55 for od in W_core}

### Procedure

In [ ]:
grids_dict = {'grid': G_grid, 'hub': G_hub, 'core': G_core}
paths_dict = {'grid': grid_paths, 'hub': hub_paths, 'core': core_paths}
W_dict = {'grid': W_grid, 'hub': W_hub, 'core': W_core}
Q_dict = {'grid': Q_grid, 'hub': Q_hub, 'core': Q_core}
cordons_dict = {'grid': cordon_grid, 'hub': cordon_hub, 'core': cordon_core}

In [ ]:
def simulate(grids_dict, paths_dict, W_dict, Q_dict,
             kappa=KAPPA, theta=THETA, max_iter=2000, tol=1e-6, damping=1.0,
             return_tolls=False):
    # Unpack grid arguments.
    G_grid, grid_paths = grids_dict['grid'], paths_dict['grid']
    W_grid, Q_grid     = W_dict['grid'], Q_dict['grid']
    cordon_grid        = cordons_dict['grid']
    # ... hub-and-core arguments.
    G_hub, hub_paths = grids_dict['hub'], paths_dict['hub']
    W_hub, Q_hub     = W_dict['hub'], Q_dict['hub']
    cordon_hub       = cordons_dict['hub']
    # ... core-periphery arguments.
    G_core, core_paths = grids_dict['core'], paths_dict['core']
    W_core, Q_core     = W_dict['core'], Q_dict['core']
    cordon_core        = cordons_dict['core']
    
    print("UNPRICED SUE")
    print("============")
    print("Grid network:           ", end='')
    x_grid, f_grid = solver.solve_sue(
        G_grid, grid_paths, W_grid, Q_grid,
        theta=theta, kappa=kappa,
        max_iter=max_iter, tol=tol, damping=damping
    )
    print("Hub-and-spoke-network:  ", end='')
    x_hub, f_hub = solver.solve_sue(
        G_hub, hub_paths, W_hub, Q_hub,
        theta=theta, kappa=kappa,
        max_iter=max_iter, tol=tol, damping=damping
    )
    print("Core-periphery network: ", end='')
    x_core, f_core = solver.solve_sue(
        G_core, core_paths, W_core, Q_core,
        theta=theta, kappa=kappa,
        max_iter=max_iter, tol=tol, damping=damping        
    )

    print("\nFIRST-BEST SUE")
    print("==============")
    print("Grid network:           ", end='')
    x_grid_fb, f_grid_fb, tolls_grid = solver.solve_sue_first_best(
        G_grid, grid_paths, W_grid, Q_grid, x_grid,
        theta=theta, kappa=kappa,
        max_iter=max_iter, tol=tol, damping=damping
    )
    print("Hub-and-spoke-network:  ", end='')
    x_hub_fb, f_hub_fb, tolls_hub = solver.solve_sue_first_best(
        G_hub, hub_paths, W_hub, Q_hub, x_hub,
        theta=theta, kappa=kappa,
        max_iter=max_iter, tol=tol, damping=damping
    )
    print("Core-periphery network: ", end='')
    x_core_fb, f_core_fb, tolls_core = solver.solve_sue_first_best(
        G_core, core_paths, W_core, Q_core, x_core,
        theta=theta, kappa=kappa,
        max_iter=max_iter, tol=tol, damping=damping
    )

    print("\nCORDON SUE")
    print("==========")
    print("Getting gordon edges...")
    cordon_edges_grid = networks.get_cordon_edges(grid_paths, cordon_grid)
    cordon_edges_hub  = networks.get_cordon_edges(hub_paths, cordon_hub)
    cordon_edges_core = networks.get_cordon_edges(core_paths, cordon_core)
    print("Computing cordon tolls...")
    cordon_toll_grid = solver.compute_cordon_toll(tolls_grid, cordon_edges_grid)
    cordon_toll_hub  = solver.compute_cordon_toll(tolls_hub, cordon_edges_hub)
    cordon_toll_core = solver.compute_cordon_toll(tolls_core, cordon_edges_core)
    print("Grid network:           ", end='')
    x_grid_sb, f_grid_sb = solver.solve_sue(
        G_grid, grid_paths, W_grid, Q_grid, 
        tau_z=cordon_toll_grid, theta=theta, kappa=kappa,
        max_iter=max_iter, tol=tol, damping=damping
    )
    print("Hub-and-spoke-network:  ", end='')
    x_hub_sb, f_hub_sb = solver.solve_sue(
        G_hub, hub_paths, W_hub, Q_hub,
        tau_z=cordon_toll_hub, theta=theta, kappa=kappa,
        max_iter=max_iter, tol=tol, damping=damping
    )
    print("Core-periphery network: ", end='')
    x_core_sb, f_core_sb = solver.solve_sue(
        G_core, core_paths, W_core, Q_core,
        tau_z=cordon_toll_core, theta=theta, kappa=kappa,
        max_iter=max_iter, tol=tol, damping=damping
    )
    print()

    x_dicts = {
        'grid': {
            'unpriced': x_grid,
            'first_best': x_grid_fb,
            'cordon': x_grid_sb
        },
        'hub': {
            'unpriced': x_hub,
            'first_best': x_hub_fb,
            'cordon': x_hub_sb
        },
        'core': {
            'unpriced': x_core,
            'first_best': x_core_fb,
            'cordon': x_core_sb
        }
    }
    f_dicts = {
        'grid': {
            'unpriced': f_grid,
            'first_best': f_grid_fb,
            'cordon': f_grid_sb
        },
        'hub': {
            'unpriced': f_hub,
            'first_best': f_hub_fb,
            'cordon': f_hub_sb
        },
        'core': {
            'unpriced': f_core,
            'first_best': f_core_fb,
            'cordon': f_core_sb
        }
    }
    tolls_dict = {'grid': tolls_grid, 'hub': tolls_hub, 'core': tolls_core}
    if return_tolls:
        return x_dicts, f_dicts, tolls_dict
    return x_dicts, f_dicts

In [ ]:
def create_mode_share_df(x_grid, x_grid_fb, x_grid_sb,
                         x_hub, x_hub_fb, x_hub_sb,
                         x_core, x_core_fb, x_core_sb):
    return pd.DataFrame([
        # Grid network.
        ['Grid', 'Unpriced', *solver.compute_mode_share(x_grid).values()],
        ['Grid', 'First-best', *solver.compute_mode_share(x_grid_fb).values()],
        ['Grid', 'Cordon', *solver.compute_mode_share(x_grid_sb).values()],
        # Hub-and-spoke network.
        ['Hub-and-spoke', 'Unpriced', *solver.compute_mode_share(x_hub).values()],
        ['Hub-and-spoke', 'First-best', *solver.compute_mode_share(x_hub_fb).values()],
        ['Hub-and-spoke', 'Cordon', *solver.compute_mode_share(x_hub_sb).values()],
        # Core-periphery network.
        ['Core-periphery', 'Unpriced', *solver.compute_mode_share(x_core).values()],
        ['Core-periphery', 'First-best', *solver.compute_mode_share(x_core_fb).values()],
        ['Core-periphery', 'Cordon', *solver.compute_mode_share(x_core_sb).values()]
    ], columns=['Network', 'Pricing', 'Car share', 'Transit share'])


def create_tsc_df(x_grid, x_grid_fb, x_grid_sb,
                  x_hub, x_hub_fb, x_hub_sb,
                  x_core, x_core_fb, x_core_sb):
    tsc_df = pd.DataFrame({
        'Network': [
            'Grid',
            'Hub-and-spoke',
            'Core-periphery'
        ],
        'TSC (unpriced)': [
            solver.total_social_cost(G_grid, x_grid),
            solver.total_social_cost(G_hub, x_hub),
            solver.total_social_cost(G_core, x_core)
        ],
        'TSC (first-best)': [
            solver.total_social_cost(G_grid, x_grid_fb),
            solver.total_social_cost(G_hub, x_hub_fb),
            solver.total_social_cost(G_core, x_core_fb)
        ],
        'TSC (cordon)': [
            solver.total_social_cost(G_grid, x_grid_sb),
            solver.total_social_cost(G_hub, x_hub_sb),
            solver.total_social_cost(G_core, x_core_sb)
        ]
    })
    tsc_df['First-best gap closed by cordon'] = (
        (tsc_df['TSC (unpriced)'] - tsc_df['TSC (cordon)']) /
        (tsc_df['TSC (unpriced)'] - tsc_df['TSC (first-best)'])
    )
    return tsc_df

In [ ]:
x_dicts, _ = simulate(grids_dict, paths_dict, W_dict, Q_dict, damping=0.5)

In [ ]:
# Grid.
x_grid    = x_dicts['grid']['unpriced']
x_grid_fb = x_dicts['grid']['first_best']
x_grid_sb = x_dicts['grid']['cordon']

# Hub-and-spoke.
x_hub    = x_dicts['hub']['unpriced']
x_hub_fb = x_dicts['hub']['first_best']
x_hub_sb = x_dicts['hub']['cordon']

# Core-periphery.
x_core    = x_dicts['core']['unpriced']
x_core_fb = x_dicts['core']['first_best']
x_core_sb = x_dicts['core']['cordon']

In [ ]:
# Plot the networks in one figure.
fig_flowmaps, axes = plt.subplots(nrows=3, ncols=3, figsize=(12, 12))
node_size = 400

# Grid.
utils.plot_flows(G_grid, grid_pos, x_grid, cordon_grid,
                 ax=axes[0][0], node_size=node_size, show_colorbar=False)
utils.plot_flows(G_grid, grid_pos, x_grid_fb, cordon_grid,
                 ax=axes[0][1], node_size=node_size, show_colorbar=False)
utils.plot_flows(G_grid, grid_pos, x_grid_sb, cordon_grid,
                 ax=axes[0][2], node_size=node_size)
# Hub-and-spoke.
utils.plot_flows(G_hub, hub_pos, x_hub, cordon_hub,
                 ax=axes[1][0], node_size=node_size, show_colorbar=False)
utils.plot_flows(G_hub, hub_pos, x_hub_fb, cordon_hub,
                 ax=axes[1][1], node_size=node_size, show_colorbar=False)
utils.plot_flows(G_hub, hub_pos, x_hub_sb, cordon_hub,
                 ax=axes[1][2], node_size=node_size)
# Core-periphery.
utils.plot_flows(G_core, core_pos, x_core, cordon_core,
                 ax=axes[2][0], node_size=node_size, show_colorbar=False)
utils.plot_flows(G_core, core_pos, x_core_fb, cordon_core,
                 ax=axes[2][1], node_size=node_size, show_colorbar=False)
utils.plot_flows(G_core, core_pos, x_core_sb, cordon_core,
                 ax=axes[2][2], node_size=node_size)

row_labels = ['Grid', 'Hub-and-spoke', 'Core-periphery']
col_labels = ['Unpriced', 'Edge tolling', 'Cordon tolling']

for ax, row_label in zip(axes[:,0], row_labels):
    ax.set_ylabel(row_label, size='x-large', labelpad=20)
for ax, col_label in zip(axes[0], col_labels):
    ax.set_title(col_label, size='x-large', pad=15)

fig_flowmaps.tight_layout()
# fig_flowmaps.savefig(IMAGES / 'flow_maps.png')

In [ ]:
# Compute mode shares.
mode_share_df = create_mode_share_df(x_grid, x_grid_fb, x_grid_sb,
                                     x_hub, x_hub_fb, x_hub_sb,
                                     x_core, x_core_fb, x_core_sb)
# mode_share_df.to_csv(RESULTS / f'mode_shares_theta-{THETA}.csv', index=False)

# Compute TSC.
tsc_df = create_tsc_df(x_grid, x_grid_fb, x_grid_sb,
                       x_hub, x_hub_fb, x_hub_sb,
                       x_core, x_core_fb, x_core_sb)
# tsc_df.to_csv(RESULTS / f'tsc_theta-{THETA}.csv', index=False)

In [ ]:
print(mode_share_df)

In [ ]:
print(tsc_df)

## Parameter sensitivity

In [ ]:
mode_share_dfs = []
tsc_dfs = []

thetas = [0.5, 2.0]
for theta in thetas:
    print(f"THETA = {theta}")
    print("================")

    x_dicts, _ = simulate(grids_dict, paths_dict, W_dict, Q_dict,
                          theta=theta, damping=0.5)
    # Grid.
    x_grid    = x_dicts['grid']['unpriced']
    x_grid_fb = x_dicts['grid']['first_best']
    x_grid_sb = x_dicts['grid']['cordon']
    # Hub-and-spoke.
    x_hub    = x_dicts['hub']['unpriced']
    x_hub_fb = x_dicts['hub']['first_best']
    x_hub_sb = x_dicts['hub']['cordon']
    # Core-periphery.
    x_core    = x_dicts['core']['unpriced']
    x_core_fb = x_dicts['core']['first_best']
    x_core_sb = x_dicts['core']['cordon']
    
    # Data frames. 
    mode_share_df = create_mode_share_df(x_grid, x_grid_fb, x_grid_sb,
                                         x_hub, x_hub_fb, x_hub_sb,
                                         x_core, x_core_fb, x_core_sb)
    tsc_df = create_tsc_df(x_grid, x_grid_fb, x_grid_sb,
                           x_hub, x_hub_fb, x_hub_sb,
                           x_core, x_core_fb, x_core_sb)
    mode_share_dfs.append(mode_share_df.copy())
    tsc_dfs.append(tsc_df.copy())

    # mode_share_df.to_csv(RESULTS / f'mode_share_theta-{theta}.csv', index=False)
    # tsc_df.to_csv(RESULTS / f'tsc_theta-{theta}.csv', index=False)

In [ ]:
print("theta = 0.5")
print(mode_share_dfs[0])

In [ ]:
print("theta = 2.0")
print(mode_share_dfs[1])

In [ ]:
print("theta = 0.5")
print(tsc_dfs[0])

In [ ]:
print("theta = 2.0")
print(tsc_dfs[1])